# NME 2: Visualizing Data

During the last NME, we went over how to obtain/import data using CSVs and how to leverage the pandas and NumPy Python libraries in order to explore datasets.

In a typical data science workflow, the next step will be to visualize the data. Visualizing data is important because it helps us identify patterns and insights in our data. Today, we'll go over how to use the Matplotlib and Seaborn libraries to create data visualizations such as plot and tables.

> **Goal for this notebook:** Create plots & visualizations using Matplotlib & Seaborn

## What are Matplotlib and Seaborn?

Matplotlib and Seaborn are two of the most popular Python libraries to create data visualizations. Matplotlib is utlizied for making basic graphs such as bar graphs, histograms, pie charts, scatterplots, etc. It works efficiently with pandas dataframes and arrays and gives you control over nearly every aspect of making a graph. Seaborn is built on top of Matplotlib and has much easier syntax to create visualizations with less code.

In this notebook, we will be using Seaborn as the primary library for creating data visualizations while relying on Matplotlib for customization.

## Setup
We will be creating data visualizations using NFL data. Instead of downloading a CSV dataset, we will use another Python library called *nflreadpy*, which provides easy access to NFL data. Importing a library allows us to load directly through Python and reduce the number of files we need.

Since nflreadpy is not preinstalled in Google Colab, we will need to manually install it using the following command.

*Note: Since nflreadpy does not allow for use with pandas by default, we will also need to install pyarrow. Don't worry, we won't use it directly!*

In [ ]:
%pip install nflreadpy
%pip install pyarrow

With nflreadpy now installed, we will import the Python libraries we will be using in our coding evironment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl

For today's visualizations, we'll use three types of NFL data: season-level, weekly, and play-by-play data. Therefore, we will need to load the NFL data from nflreadpy and convert it to pandas dataframes.

In [ ]:
# Load 2025 player statistics
players = nfl.load_player_stats(
    [2025], summary_level="reg"
).to_pandas()

# Load weekly player statistics
weekly = nfl.load_player_stats(
    [2025], summary_level="week"
).to_pandas()

# Load play-by-play data
pbp = nfl.load_pbp([2025]).to_pandas()

If you're wondering how I knew which commands to use to load the player statistics, I encourage you to check out the nflreadpy documentation. Documentation is an important resource when coding because programming isn't about memorizing every command. Instead, it's about knowing how to find and use the information you need. Whenever you're learning a new library, its documentation is usually the best place to start!

## Basic Visualizations

We will go over how to use Seaborn to create 5 basic visualizations: bar charts, scatterplots, line graphs, histograms, & boxplots. Additionally, we will use Matplotlib to further customize our visualizations.

### Bar chart

**Question:** Which players rushed for the most yards in the 2025 season?

We use `sns.barplot()` to create a bar chart. We can also customize the chart using functions like `plt.title()`, `plt.xlabel()`, and `plt.ylabel()` to add a title and axis labels, making it easier to read and understand.

In [ ]:
# Find the top 10 running backs by rushing yards
top_rushers = players[players["position"] == "RB"]
top_rushers = top_rushers.nlargest(10, "rushing_yards")



### Scatterplot

**Question:** Do quarterbacks who attempt more passes usually throw for more yards?

We use `sns.scatterplot()` to create a scatterplot.

In [ ]:
# Keep quarterbacks with at least 100 attempts
qbs = players[players["position"] == "QB"]
qbs = qbs[qbs["attempts"] >= 100]



### Line Plot

**Question:** How did the top receiver’s yardage change throughout the season?

We use `sns.lineplot()` to create a line plot.

In [ ]:
# Find the receiver with the most 2025 receiving yards
receivers = players[players["position"] == "WR"]
top_receiver = receivers.nlargest(1, "receiving_yards").iloc[0]

# Get that player's weekly stats
receiver_weeks = weekly[weekly["player_id"] == top_receiver["player_id"]]
receiver_weeks = receiver_weeks.sort_values("week")

*Note: `nlargest(1, "receiving_yards")` finds the player with the most receiving yards, while `.iloc[0]` selects that player's row.*

### Histogram

**Question:** How many yards does an NFL rushing attempt typically gain?

We use `sns.histogram()` to create a histogram.

In [ ]:
# Keep rushing plays with recorded yardage
rush_plays = pbp[pbp["play_type"] == "run"]
rush_plays = rush_plays[rush_plays["rush_attempt"] == 1]
rush_plays = rush_plays[rush_plays["two_point_attempt"] != 1]


### Box Plot

**Question:** How does pass rate change with down and distance?

We use `sns.boxplot()` to create a box plot.

In [ ]:
# Find the two leading running backs
best_two = top_rushers.head(2)

# Get their individual carries
rb_runs = rush_plays[rush_plays["rusher_player_id"].isin(best_two["player_id"])].copy()

# Add readable player names
names = dict(zip(best_two["player_id"], best_two["player_display_name"]))
rb_runs["player"] = rb_runs["rusher_player_id"].map(names)



#### Wrap-up
| Chart | Best for | Seaborn function |
|---|---|---|
| Bar | Comparing categories | `sns.barplot()` |
| Scatter | Finding relationships | `sns.scatterplot()` |
| Line | Tracking changes by week | `sns.lineplot()` |
| Histogram | Showing a distribution | `sns.histplot()` |
| Box | Comparing distributions | `sns.boxplot()` |

## Advanced Visualizations

In this section, we'll explore four more advanced visualizations: bubble charts, heatmaps, and football field plots. These visualizations require a bit more data preparation and filtering using pandas, but they allow us to explore our data in more interesting ways.

### Bubble Chart

**Question:** Which teams combine a high success rate with high EPA per play?

We use `sns.scatterplot()` to create a bubble chart by utilizing the `size` paramter.


In [ ]:
# Select wide receivers
wrs = players[players["position"] == "WR"]



## Heatmap

**Question:** Which quarterback statistics move together?

We use sns.heatmap() to create the heatmap and .corr() to calculate the correlations between numerical variables. Together, these allow us to create a correlation heatmap.

In [ ]:
# Select running backs
rbs = players[players["position"] == "RB"]

# Choose statistics to compare
stats = rbs[[
    "carries",
    "rushing_yards",
    "rushing_tds",
    "targets",
    "receiving_yards",
    "receiving_tds"
]]



### Football Field Plot

We use Matplotlib functions like `plt.axvspan()` to create the football field and `plt.axvline()` to draw the yard lines. We then use `sns.scatterplot()` to plot the rushing touchdowns on the field based on their starting positions.

In [ ]:
# Find rushing touchdowns
run_tds = pbp[
    (pbp["play_type"] == "run") &
    (pbp["rush_touchdown"] == 1)
].dropna(subset=["yardline_100"]).copy()

# Calculate field positions
run_tds["field_x"] = 110 - run_tds["yardline_100"]
run_tds["field_y"] = np.random.choice(
    [23.6, 26.65, 29.7],
    size=len(run_tds)) # randomly generate y-coord since nflreadpy doesn't include it

# Create football field
plt.figure(figsize=(12, 5))
plt.axvspan(0, 120, color="forestgreen")

# Draw yard lines
for x in range(10, 111, 10):
    plt.axvline(x, color="white", alpha=0.5)

# Plot rushing touchdown locations
sns.scatterplot(
    data=run_tds,
    x="field_x",
    y="field_y",
    color="gold",
    s=70
)

plt.xlim(0, 120)
plt.ylim(0, 53.3)
plt.title("Rushing Touchdown Starting Positions (2025)")
plt.xlabel("Field Position")
plt.yticks([])
plt.show()

#### Wrap-up

| Chart | Best for | Function |
|---|---|---|
| Bubble | Comparing three numerical variables | `sns.scatterplot()` |
| Heatmap | Finding correlations between variables | `sns.heatmap()` + `.corr()` |
| Football Field | Visualizing play locations on a field | `plt.axvspan()`, `plt.axvline()`, `sns.scatterplot()` |